# MangoScan Model 0 — Mango/Stem Detector (Kaggle)

Kaggle variant of `MangoScan_Detector_Colab.ipynb`. Nothing lives on your
laptop: both the dataset and the code are uploaded as Kaggle Datasets.

## Before you run a single cell

Open the **Notebook options** panel on the right and set:

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** or **GPU P100** |
| Internet | **On** |

**Internet is not optional and it is the step people get stuck on.** Without
it `pip install ultralytics` fails *and* the pretrained `yolo11n.pt`
checkpoint cannot download. Kaggle requires a **verified phone number**
before it will let you switch internet on — do that first, at
Settings → Phone Verification.

## Two datasets to upload

Create both at *Datasets → New Dataset*:

| Upload | From | Suggested slug |
|---|---|---|
| the prepared data, ~600 MB | `MyTasks/data/detector_dataset/` | `mangoscan-detector-dataset` |
| the code, tiny | `MyTasks/model0_detector/` | `mangoscan-model0-code` |

Then *Add Input* on this notebook and attach both.

Kaggle gives 30 GPU-hours/week and a 12-hour session ceiling. This trains in
well under both.

In [ ]:
!nvidia-smi

## 0. Find the two inputs and sanity-check the dataset

Kaggle slugs vary with how you named the upload, so this searches rather
than hardcoding. If either assert fires, check *Add Input* on the right.

If image and label counts disagree for any split, stop — Ultralytics trains
happily on a mismatched pair and the metrics come out quietly wrong.

In [ ]:
import pathlib

INPUT = pathlib.Path('/kaggle/input')
WORK = pathlib.Path('/kaggle/working')

print('attached inputs:', [p.name for p in INPUT.iterdir()])


def find(marker, label):
    """Locate the dataset directory containing `marker`, at any nesting depth.

    Kaggle wraps an uploaded folder in an extra directory sometimes and not
    others, depending on whether you uploaded a zip or a folder.
    """
    hits = sorted(INPUT.glob(f'*/**/{marker}')) + sorted(INPUT.glob(f'*/{marker}'))
    assert hits, f'Could not find {label} (looked for {marker!r}). Use Add Input.'
    return hits[0].parent


DATA = find('data.yaml', 'the detector dataset')
CODE = find('configs/detector.toml', 'the model0_detector code')
print('\ndata:', DATA)
print('code:', CODE)

print()
for split in ('train', 'val', 'test'):
    n_img = len(list((DATA / 'images' / split).glob('*.jpg')))
    n_lbl = len(list((DATA / 'labels' / split).glob('*.txt')))
    flag = '' if n_img == n_lbl else '   <-- MISMATCH, stop here'
    print(f'{split:<6} {n_img:>5} images  {n_lbl:>5} labels{flag}')

## 1. Install, and copy the code somewhere writable

`/kaggle/input` is **read-only**. Training writes checkpoints on every
epoch, so the code and all outputs have to live under `/kaggle/working`.

In [ ]:
!pip install -q ultralytics

import shutil

local = WORK / 'model0_detector'
if local.exists():
    shutil.rmtree(local)
shutil.copytree(CODE, local)

%cd /kaggle/working/model0_detector
!ls -R | head -20

## 2. Write a `data.yaml` that points at `/kaggle/input`

The uploaded `data.yaml` carries an absolute Windows `path` from the machine
that prepared the dataset, and it sits on a read-only volume so it cannot be
patched in place. Write a fresh one into the working directory instead — the
images stay where they are and are only read.

In [ ]:
DATA_YAML = WORK / 'data.yaml'
DATA_YAML.write_text(
    f'path: {DATA.as_posix()}\n'
    'train: images/train\n'
    'val: images/val\n'
    'test: images/test\n'
    'nc: 2\n'
    'names:\n'
    '  0: mango\n'
    '  1: stem\n'
)
print(DATA_YAML.read_text())

## 3. Train

150 epochs over ~1,700 images takes roughly 1.5–3 hours on a P100. Early
stopping fires after 30 epochs without improvement, so it usually finishes
sooner. Well inside Kaggle's 12-hour session limit.

Watch `metrics/recall(B)` more than the loss curves — see the note at the
top of `scripts/03_evaluate.py` for why recall is the number that matters.

`--device 0` uses one GPU. If you picked **T4 x2**, `--device 0,1` is
available but rarely worth it at this dataset size — the dataloader becomes
the bottleneck before the GPUs do.

In [ ]:
!python scripts/02_train.py --config configs/detector.toml \
    --data /kaggle/working/data.yaml \
    --device 0

## 4. Evaluate on the held-out test split

**mango recall** is the number to defend. A missed mango is a fruit that
never reaches Models 1–4 and never gets routed at all — strictly worse than
one routed to the wrong bin.

Stem numbers will be lower, and that is expected: the stem is small, often
occluded, and supplies orientation only. It is not on the routing path, so
do not tune against it.

In [ ]:
!python scripts/03_evaluate.py --config configs/detector.toml \
    --weights runs/model0_detector/weights/best.pt \
    --data /kaggle/working/data.yaml

## 5. Look at actual predictions

Metrics hide failure modes. Look at the boxes. You are checking for two
things: mangoes with no box at all, and boxes loose enough to include
background — Model 4 reads colour off exactly those pixels.

In [ ]:
import random

import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO('runs/model0_detector/weights/best.pt')
test_images = sorted((DATA / 'images' / 'test').glob('*.jpg'))
sample = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img in zip(axes.ravel(), sample):
    result = model.predict(str(img), imgsz=640, conf=0.25, verbose=False)[0]
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f'{img.name}  ({len(result.boxes)} boxes)', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Export for deployment

ONNX is what the rig actually runs. This is the artefact that matters —
and the reason this is worth doing on Kaggle rather than on a hosted
platform that keeps the weights.

In [ ]:
from ultralytics import YOLO

model = YOLO('runs/model0_detector/weights/best.pt')
model.export(format='onnx', opset=12, simplify=True, imgsz=640)

## 7. Collect the artefacts, then **Save Version**

This copies the weights to the top of `/kaggle/working` so they are easy to
find in the output browser.

**Then click *Save Version* (top right) and wait for it to finish.** Kaggle
discards the session container when it stops; only files persisted by a
saved version survive, and that is where you download `best.pt` and
`best.onnx` from. Losing a 3-hour run to a closed tab is a real way to lose
an afternoon.

In [ ]:
import pathlib
import shutil

dest = pathlib.Path('/kaggle/working/model0_detector_weights')
dest.mkdir(parents=True, exist_ok=True)
src = pathlib.Path('/kaggle/working/model0_detector/runs/model0_detector/weights')

for f in sorted(src.glob('*.pt')) + sorted(src.glob('*.onnx')) + sorted(src.glob('*.json')):
    shutil.copy2(f, dest / f.name)
    print(f'{f.name:<28} {f.stat().st_size / 1e6:>7.1f} MB')

print(f'\ncollected in {dest}')
print('Now click Save Version, or these are gone when the session ends.')